# NB07c — Weather Risk Index (score_weather)

Fetches **1 year of KMA hourly observations** (2025 full year, station 수원 STN=119)
in ≤31-day chunks and computes drone weather availability for every H3 cell.

## Score definition

```
P_drone_weather_available = (1 - P_wind_fail) × (1 - P_rain_fail) × (1 - P_visibility_fail)
weather_risk  = 1 - P_drone_weather_available
score_weather = P_drone_weather_available   ← continuous 0–1 input for NB09
```

**Drone failure thresholds:** wind ≥ 10 m/s · rain ≥ 5 mm/h · visibility 3% default

## Why station-level?

기상 데이터는 격자/관측소 단위입니다. 모든 H3 셀이 동일한 `score_weather`를 받습니다.
NB09에서 공간적 차별화는 `Ra`(로봇 접근성) 등 다른 레이어가 담당합니다.

`score_weather`는 연속 점수(soft score)입니다 — NB09의 이진 필터가 아닙니다.

## Outputs
- `processed/weather_history.csv` — cleaned 1-year hourly observations
- `processed/weather_observed_summary.json` — failure probability summary
- `processed/weather_risk_grid.gpkg` / `.csv` — H3 grid with score_weather
- `processed/weather_risk_chart.png` — diagnostics chart

In [1]:
import os
import json
import time
import warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

BASE = Path('..').resolve()
PROC = BASE / 'processed'

print(f'Project root : {BASE}')
print(f'Processed    : {PROC}')
print(f'Run time     : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


Project root : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset
Processed    : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed
Run time     : 2026-05-10 11:33:10


In [2]:
_KEY_FROM_ENV = False
KMA_AUTH_KEY  = os.getenv('KMA_API_KEY') or os.getenv('KMA_AUTH_KEY')
if KMA_AUTH_KEY:
    _KEY_FROM_ENV = True
    print(f'KMA key source : environment variable  (len={len(KMA_AUTH_KEY)})')
else:
    KMA_AUTH_KEY = '8FPiv31cRI6T4r99XOSOCA'
    print(f'KMA key source : local fallback  (len={len(KMA_AUTH_KEY)})')
print(f'Key preview    : {KMA_AUTH_KEY[:4]}...{KMA_AUTH_KEY[-4:]}')


KMA key source : local fallback  (len=22)
Key preview    : 8FPi...SOCA


In [3]:
# Weather period — full calendar year 2025
# Aligns with the reset project's 2025 demand and parking data.
START_DATE = '2025-01-01'
END_DATE   = '2025-12-31'

# Drone weather failure thresholds
WIND_FAIL_MS     = 10.0   # m/s
RAIN_FAIL_MMH    =  5.0   # mm/h
DEFAULT_VIS_FAIL =  0.03  # 3 % assumed when KMA vis column unavailable

# KMA station
STATION_ID   = 119
STATION_NAME = '수원(119)'

# Chunk size: KMA kma_sfctm3.php allows ≤ 31 days per request
CHUNK_DAYS   = 31

# KMA endpoint
KMA_URL = 'https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php'

print(f'Weather period : {START_DATE}  ~  {END_DATE}')
print(f'Station        : {STATION_NAME}  (STN={STATION_ID})')
print(f'Chunk size     : ≤ {CHUNK_DAYS} days per request')
print(f'Thresholds     : wind ≥ {WIND_FAIL_MS} m/s  |  rain ≥ {RAIN_FAIL_MMH} mm/h  |  vis default {DEFAULT_VIS_FAIL*100:.0f}%')


Weather period : 2025-01-01  ~  2025-12-31
Station        : 수원(119)  (STN=119)
Chunk size     : ≤ 31 days per request
Thresholds     : wind ≥ 10.0 m/s  |  rain ≥ 5.0 mm/h  |  vis default 3%


In [4]:
demand_gpkg = PROC / 'demand_grid.gpkg'
demand_csv  = PROC / 'demand_grid.csv'

if not demand_gpkg.exists():
    raise FileNotFoundError(
        f'demand_grid.gpkg not found at {demand_gpkg}. '
        'Run NB08_demand_grid_index.ipynb first.'
    )

hex_gdf = gpd.read_file(demand_gpkg)
print(f'H3 grid: {len(hex_gdf)} cells  CRS={hex_gdf.crs}')

if 'h3_index' not in hex_gdf.columns and 'h3_id' in hex_gdf.columns:
    hex_gdf['h3_index'] = hex_gdf['h3_id']

REQUIRED = ['h3_index', 'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD', 'Ds', 'demand_grade']
missing_gpkg = [c for c in REQUIRED if c not in hex_gdf.columns]
if missing_gpkg and demand_csv.exists():
    dcsv = pd.read_csv(demand_csv, encoding='utf-8-sig',
                       usecols=lambda c: c in REQUIRED)
    hex_gdf = hex_gdf.merge(dcsv, on='h3_index', how='left', suffixes=('', '_csv'))
    for col in missing_gpkg:
        if f'{col}_csv' in hex_gdf.columns:
            hex_gdf[col] = hex_gdf[f'{col}_csv']
            hex_gdf.drop(columns=[f'{col}_csv'], inplace=True)
    print(f'  Pulled from CSV: {missing_gpkg}')

print()
print('Required column null counts:')
for col in REQUIRED:
    if col in hex_gdf.columns:
        nn = hex_gdf[col].isna().sum()
        print(f'  {col}: {nn} nulls{"  <-- PROBLEM" if nn > 0 else ""}')
    else:
        print(f'  {col}: COLUMN MISSING')


H3 grid: 283 cells  CRS=EPSG:4326

Required column null counts:
  h3_index: 0 nulls
  lat: 0 nulls
  lon: 0 nulls
  ADM_NM: 0 nulls
  GU_NM: 0 nulls
  CSV_ADMI_CD: 0 nulls
  Ds: 0 nulls
  demand_grade: 0 nulls


In [5]:
def _make_chunks(start_str, end_str, chunk_days):
    '''Split [start, end] into ≤ chunk_days-day intervals.'''
    start = datetime.strptime(start_str, '%Y-%m-%d')
    end   = datetime.strptime(end_str,   '%Y-%m-%d').replace(hour=23, minute=59)
    chunks = []
    cur = start
    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days) - timedelta(minutes=1), end)
        chunks.append((cur, chunk_end))
        cur = chunk_end + timedelta(minutes=1)
    return chunks

chunks = _make_chunks(START_DATE, END_DATE, CHUNK_DAYS)
print(f'Total chunks: {len(chunks)}')
for i, (s, e) in enumerate(chunks):
    print(f'  Chunk {i+1:2d}: {s.strftime("%Y-%m-%d")} ~ {e.strftime("%Y-%m-%d")}')


Total chunks: 12
  Chunk  1: 2025-01-01 ~ 2025-01-31
  Chunk  2: 2025-02-01 ~ 2025-03-03
  Chunk  3: 2025-03-04 ~ 2025-04-03
  Chunk  4: 2025-04-04 ~ 2025-05-04
  Chunk  5: 2025-05-05 ~ 2025-06-04
  Chunk  6: 2025-06-05 ~ 2025-07-05
  Chunk  7: 2025-07-06 ~ 2025-08-05
  Chunk  8: 2025-08-06 ~ 2025-09-05
  Chunk  9: 2025-09-06 ~ 2025-10-06
  Chunk 10: 2025-10-07 ~ 2025-11-06
  Chunk 11: 2025-11-07 ~ 2025-12-07
  Chunk 12: 2025-12-08 ~ 2025-12-31


In [6]:
def _safe_float(s, sentinel_to_zero=False):
    '''Parse a token to float. Sentinel < -1 → 0 when sentinel_to_zero is True.'''
    if s is None:
        return np.nan
    s = str(s).strip()
    if s in ('', '-', 'None', 'nan'):
        return np.nan
    try:
        f = float(s)
        if sentinel_to_zero and f < -1:
            return 0.0
        return f
    except ValueError:
        return np.nan

def _parse_lines(lines, chunk_start_str, chunk_end_str):
    '''Parse kma_sfctm3 space-delimited lines into a list of dicts.'''
    records = []
    for line in lines:
        parts = line.split()
        if len(parts) < 4:
            continue
        try:
            rec = {
                'tm'  : parts[0],
                'stn' : parts[1],
                'wd'  : _safe_float(parts[2] if len(parts) > 2  else None),
                'ws'  : _safe_float(parts[3] if len(parts) > 3  else None),
                'ta'  : _safe_float(parts[11] if len(parts) > 11 else None),
                'rn'  : _safe_float(parts[15] if len(parts) > 15 else None,
                                    sentinel_to_zero=True),
                'source_station' : STATION_NAME,
                'chunk_start'    : chunk_start_str,
                'chunk_end'      : chunk_end_str,
            }
            # Visibility: probe common column positions
            vis_val = None
            for vi in [20, 21, 19, 22]:
                if len(parts) > vi:
                    v = _safe_float(parts[vi])
                    if pd.notna(v) and 0 <= v <= 100_000:
                        vis_val = v
                        break
            rec['vis'] = vis_val
            records.append(rec)
        except Exception:
            pass
    return records

all_records   = []
n_chunks      = len(chunks)
ok_chunks     = 0
fail_chunks   = 0
fail_details  = []

print(f'Fetching {n_chunks} chunks from KMA ({START_DATE} ~ {END_DATE}) ...')
print()

for i, (cs, ce) in enumerate(chunks):
    tm1 = cs.strftime('%Y%m%d%H%M')
    tm2 = ce.strftime('%Y%m%d%H%M')
    url = (f'{KMA_URL}?tm1={tm1}&tm2={tm2}'
           f'&stn={STATION_ID}&authKey={KMA_AUTH_KEY}')
    try:
        resp = requests.get(url, timeout=40)
        resp.encoding = 'utf-8'
        lines = [l for l in resp.text.split('\n')
                 if l.strip() and not l.strip().startswith('#') and len(l.strip()) > 10]
        recs = _parse_lines(lines, tm1, tm2)
        if recs:
            all_records.extend(recs)
            ok_chunks += 1
            print(f'  Chunk {i+1:2d}/{n_chunks}: {cs.strftime("%Y-%m-%d")} ~ {ce.strftime("%Y-%m-%d")}'
                  f'  →  {len(recs):4d} rows  [OK]')
        else:
            fail_chunks += 1
            fail_details.append(f'Chunk {i+1}: empty response')
            print(f'  Chunk {i+1:2d}/{n_chunks}: {cs.strftime("%Y-%m-%d")} ~ {ce.strftime("%Y-%m-%d")}'
                  f'  →  empty  [WARN]')
    except Exception as e:
        fail_chunks += 1
        fail_details.append(f'Chunk {i+1}: {e}')
        print(f'  Chunk {i+1:2d}/{n_chunks}: FAILED — {e}')
    # Polite pause between requests
    if i < n_chunks - 1:
        time.sleep(0.5)

print()
print(f'Chunks succeeded : {ok_chunks}/{n_chunks}')
print(f'Chunks failed    : {fail_chunks}/{n_chunks}')
print(f'Total records    : {len(all_records):,}')
if fail_details:
    for d in fail_details:
        print(f'  [FAIL] {d}')


Fetching 12 chunks from KMA (2025-01-01 ~ 2025-12-31) ...



  Chunk  1/12: 2025-01-01 ~ 2025-01-31  →   744 rows  [OK]


  Chunk  2/12: 2025-02-01 ~ 2025-03-03  →   744 rows  [OK]


  Chunk  3/12: 2025-03-04 ~ 2025-04-03  →   744 rows  [OK]


  Chunk  4/12: 2025-04-04 ~ 2025-05-04  →   744 rows  [OK]


...

  Chunk  8/12: 2025-08-06 ~ 2025-09-05  →   744 rows  [OK]


  Chunk  9/12: 2025-09-06 ~ 2025-10-06  →   744 rows  [OK]


  Chunk 10/12: 2025-10-07 ~ 2025-11-06  →   744 rows  [OK]


  Chunk 11/12: 2025-11-07 ~ 2025-12-07  →   744 rows  [OK]


  Chunk 12/12: 2025-12-08 ~ 2025-12-31  →   576 rows  [OK]

Chunks succeeded : 12/12
Chunks failed    : 0/12
Total records    : 8,760


In [7]:
if not all_records:
    raise RuntimeError(
        'No weather data fetched from KMA.\n'
        'Check network access, auth key, and date range.'
    )

raw_df = pd.DataFrame(all_records)

# Convert tm to datetime
raw_df['dt'] = pd.to_datetime(raw_df['tm'], format='%Y%m%d%H%M', errors='coerce')

# Numeric conversions
for col in ['ws', 'ta']:
    raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')
raw_df['rn'] = pd.to_numeric(raw_df['rn'], errors='coerce').fillna(0.0)

# Count and replace any remaining negative rainfall sentinels
neg_rain_mask = raw_df['rn'] < 0
n_neg_rain = int(neg_rain_mask.sum())
raw_df.loc[neg_rain_mask, 'rn'] = 0.0

# Deduplicate on tm + stn, keep first occurrence
before_dedup = len(raw_df)
hist_df = raw_df.drop_duplicates(subset=['tm', 'stn'], keep='first').sort_values('dt').reset_index(drop=True)
n_dropped_dup = before_dedup - len(hist_df)

print(f'Raw rows (all chunks) : {before_dedup:,}')
print(f'Duplicate tm rows     : {n_dropped_dup:,}')
print(f'After dedup           : {len(hist_df):,}')
print(f'Negative rain → 0     : {n_neg_rain:,}')
print()

n_missing_ws = hist_df['ws'].isna().sum()
n_vis_avail  = hist_df['vis'].notna().sum() if 'vis' in hist_df.columns else 0

print(f'Missing wind speed    : {n_missing_ws:,}')
print(f'Visibility records    : {n_vis_avail:,}')
print()
print('Wind speed (m/s):')
print(hist_df['ws'].dropna().describe().round(3).to_string())
print()
print('Rainfall (mm/h):')
print(hist_df['rn'].describe().round(4).to_string())


Raw rows (all chunks) : 8,760
Duplicate tm rows     : 0
After dedup           : 8,760
Negative rain → 0     : 0

Missing wind speed    : 0
Visibility records    : 2,129

Wind speed (m/s):
count    8760.000
mean        1.970
std         1.232
min         0.000
25%         1.000
50%         1.800
75%         2.800
max         8.700

Rainfall (mm/h):
count    8760.0000
mean        0.1444
std         1.1018
min         0.0000
25%         0.0000
50%         0.0000
75%         0.0000
max        32.1000


In [8]:
# Save cleaned history (convert datetime to string for CSV portability)
hist_save = hist_df.copy()
hist_save['dt'] = hist_save['dt'].astype(str)
hist_path = PROC / 'weather_history.csv'
hist_save.to_csv(hist_path, index=False, encoding='utf-8-sig')
print(f'Saved: {hist_path.name}  ({len(hist_save):,} rows  {hist_path.stat().st_size // 1024} KB)')


Saved: weather_history.csv  (8,760 rows  816 KB)


In [9]:
# Use rows with valid wind speed for wind/rain probability
ws_valid = hist_df.dropna(subset=['ws'])
n_obs    = len(ws_valid)

# Wind failure
wind_fail_mask = ws_valid['ws'] >= WIND_FAIL_MS
n_wind_fail    = int(wind_fail_mask.sum())
P_wind_fail    = n_wind_fail / n_obs

# Rain failure (rn has been cleaned — no negatives)
rain_fail_mask = ws_valid['rn'] >= RAIN_FAIL_MMH
n_rain_fail    = int(rain_fail_mask.sum())
P_rain_fail    = n_rain_fail / n_obs

# Visibility failure
n_vis = int(hist_df['vis'].notna().sum()) if 'vis' in hist_df.columns else 0
if n_vis >= 24:
    # KMA kma_sfctm3 reports vis in km (values 0–97); threshold < 1.0 km
    P_vis_fail = float((hist_df['vis'].dropna() < 1.0).sum() / n_vis)
    vis_note   = f'observed ({n_vis} records, threshold < 1.0 km)'
else:
    P_vis_fail = DEFAULT_VIS_FAIL
    vis_note   = f'default {DEFAULT_VIS_FAIL*100:.0f}% (visibility data insufficient: {n_vis} records)'

# Drone availability
P_drone_ok    = (1 - P_wind_fail) * (1 - P_rain_fail) * (1 - P_vis_fail)
weather_risk  = float(np.clip(1 - P_drone_ok, 0, 1))
score_weather = float(np.clip(P_drone_ok, 0, 1))

period_start  = str(hist_df['dt'].min())[:10]
period_end    = str(hist_df['dt'].max())[:10]

print('[Failure Probabilities]')
print(f'  Valid wind obs    : {n_obs:,}  ({period_start} ~ {period_end})')
print(f'  P_wind_fail       : {P_wind_fail*100:.3f}%  ({n_wind_fail}/{n_obs} hrs ≥ {WIND_FAIL_MS} m/s)')
print(f'  P_rain_fail       : {P_rain_fail*100:.3f}%  ({n_rain_fail}/{n_obs} hrs ≥ {RAIN_FAIL_MMH} mm/h)')
print(f'  P_vis_fail        : {P_vis_fail*100:.2f}%  [{vis_note}]')
print(f'  P_drone_ok        : {P_drone_ok*100:.3f}%')
print(f'  weather_risk      : {weather_risk:.6f}')
print(f'  score_weather     : {score_weather:.6f}')


[Failure Probabilities]
  Valid wind obs    : 8,760  (2025-01-01 ~ 2025-12-31)
  P_wind_fail       : 0.000%  (0/8760 hrs ≥ 10.0 m/s)
  P_rain_fail       : 0.742%  (65/8760 hrs ≥ 5.0 mm/h)
  P_vis_fail        : 4.88%  [observed (2129 records, threshold < 1.0 km)]
  P_drone_ok        : 94.409%
  weather_risk      : 0.055907
  score_weather     : 0.944093


In [10]:
summary = {
    'weather_source'             : 'kma_hourly_observed_1year',
    'station_id'                 : STATION_ID,
    'station_name'               : STATION_NAME,
    'observation_count'          : n_obs,
    'weather_period_start'       : period_start,
    'weather_period_end'         : period_end,
    'chunk_count'                : n_chunks,
    'chunk_success_count'        : ok_chunks,
    'chunk_fail_count'           : fail_chunks,
    'wind_fail_threshold_ms'     : WIND_FAIL_MS,
    'rain_fail_threshold_mmh'    : RAIN_FAIL_MMH,
    'visibility_fail_note'       : vis_note,
    'P_wind_fail'                : round(float(P_wind_fail),  6),
    'P_rain_fail'                : round(float(P_rain_fail),  6),
    'P_visibility_fail'          : round(float(P_vis_fail),   6),
    'P_drone_weather_available'  : round(float(P_drone_ok),   6),
    'weather_risk'               : round(float(weather_risk), 6),
    'score_weather'              : round(float(score_weather),6),
    'rain_negative_value_rule'   : 'values < -1 treated as 0 (KMA sentinel for no-rainfall)',
    'notes': (
        'score_weather is station-level (STN=119); all H3 cells receive the same value. '
        'Spatial differentiation within Seongnam is provided by Ra and other NB07 layers. '
        'NB09 uses score_weather as a continuous soft input, not a binary filter.'
    ),
}
# Key is not included in the output file

obs_path = PROC / 'weather_observed_summary.json'
obs_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved: {obs_path.name}')
print(json.dumps(summary, ensure_ascii=False, indent=2))


Saved: weather_observed_summary.json
{
  "weather_source": "kma_hourly_observed_1year",
  "station_id": 119,
  "station_name": "수원(119)",
  "observation_count": 8760,
  "weather_period_start": "2025-01-01",
  "weather_period_end": "2025-12-31",
  "chunk_count": 12,
  "chunk_success_count": 12,
  "chunk_fail_count": 0,
  "wind_fail_threshold_ms": 10.0,
  "rain_fail_threshold_mmh": 5.0,
  "visibility_fail_note": "observed (2129 records, threshold < 1.0 km)",
  "P_wind_fail": 0.0,
  "P_rain_fail": 0.00742,
  "P_visibility_fail": 0.048849,
  "P_drone_weather_available": 0.944093,
  "weather_risk": 0.055907,
  "score_weather": 0.944093,
  "rain_negative_value_rule": "values < -1 treated as 0 (KMA sentinel for no-rainfall)",
  "notes": "score_weather is station-level (STN=119); all H3 cells receive the same value. Spatial differentiation within Seongnam is provided by Ra and other NB07 layers. NB09 uses score_weather as a continuous soft input, not a binary filter."
}


In [11]:
# Optional: fetch current ultra-short observation for Go/No-Go display.
# This does NOT influence score_weather — it is a present-moment check only.
print('[Optional] Fetching current weather (ultra-short observation) ...')

now_dt    = datetime.now()
base_date = now_dt.strftime('%Y%m%d')
base_hour = now_dt.hour if now_dt.minute >= 30 else max(now_dt.hour - 1, 0)
base_time = f'{base_hour:02d}00'

url_cur = (
    f'https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/'
    f'getUltraSrtNcst?pageNo=1&numOfRows=1000&dataType=JSON'
    f'&base_date={base_date}&base_time={base_time}'
    f'&nx=62&ny=123&authKey={KMA_AUTH_KEY}'
)

CURRENT_OK = False
current_payload = {
    'timestamp': now_dt.isoformat(),
    'base_date': base_date,
    'base_time': base_time,
    'nx': 62, 'ny': 123,
    'location': '성남시 분당구',
    'go_nogo': 'UNKNOWN',
    'note': 'does not affect 1-year score_weather',
}

try:
    rc   = requests.get(url_cur, timeout=15)
    data = rc.json()
    items = data['response']['body']['items']['item']
    obs   = {it['category']: it['obsrValue'] for it in items}

    wind_speed = float(obs.get('WSD', 0) or 0)
    rain_1h    = float(obs.get('RN1', 0) or 0)
    pty        = int(float(obs.get('PTY', 0) or 0))
    temp       = float(obs.get('T1H', 0) or 0)
    humidity   = float(obs.get('REH', 0) or 0)

    reasons = []
    if wind_speed >= WIND_FAIL_MS:
        reasons.append(f'강풍({wind_speed:.1f} m/s)')
    if rain_1h >= RAIN_FAIL_MMH:
        reasons.append(f'강우({rain_1h:.1f} mm/h)')
    if pty > 0:
        pty_lbl = {1:'비',2:'비/눈',3:'눈',4:'소나기',5:'빗방울',6:'빗방울/눈날림',7:'눈날림'}
        reasons.append(f'강수형태({pty_lbl.get(pty, str(pty))})')

    go_nogo = 'NO-GO' if reasons else 'GO'
    current_payload.update({'wind_speed': wind_speed, 'rain_1h': rain_1h,
                             'pty': pty, 'temperature': temp, 'humidity': humidity,
                             'go_nogo': go_nogo, 'reasons': reasons})
    CURRENT_OK = True
    print(f'  {base_date} {base_time}  wind={wind_speed:.1f} m/s  rain={rain_1h:.1f} mm/h  pty={pty}')
    print(f'  Current Go/No-Go: [{go_nogo}]' + (f'  {reasons}' if reasons else ''))
except Exception as e:
    current_payload['error'] = str(e)
    print(f'  Fetch failed: {e}  (non-critical — score_weather is unaffected)')

cur_path = PROC / 'weather_current.json'
cur_path.write_text(json.dumps(current_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'  Saved: {cur_path.name}')


[Optional] Fetching current weather (ultra-short observation) ...


  20260510 1100  wind=0.7 m/s  rain=0.0 mm/h  pty=0
  Current Go/No-Go: [GO]
  Saved: weather_current.json


In [12]:
WEATHER_NOTE = (
    f'station-level score: all H3 cells receive the same score_weather={score_weather:.4f}. '
    f'Source: {STATION_NAME}, {period_start}~{period_end}, {n_obs} hourly observations.'
)

grid = hex_gdf[['h3_index', 'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD',
                 'Ds', 'demand_grade', 'geometry']].copy()

grid['P_wind_fail']               = round(float(P_wind_fail),  6)
grid['P_rain_fail']               = round(float(P_rain_fail),  6)
grid['P_visibility_fail']         = round(float(P_vis_fail),   6)
grid['P_drone_weather_available'] = round(float(P_drone_ok),   6)
grid['weather_risk']              = round(float(weather_risk), 6)
grid['score_weather']             = round(float(score_weather),6)
grid['weather_station']           = STATION_NAME
grid['weather_source']            = 'kma_hourly_observed_1year'
grid['weather_period_start']      = period_start
grid['weather_period_end']        = period_end
grid['weather_note']              = WEATHER_NOTE

# ── GPKG (drop lowercase aliases to avoid OGR case conflict) ─────────────────
gpkg_gdf = grid.copy()
for col in ['gu_nm', 'adm_nm', 'dong_nm', 'h3_id']:
    if col in gpkg_gdf.columns:
        gpkg_gdf.drop(columns=[col], inplace=True)
gpkg_gdf['demand_grade'] = gpkg_gdf['demand_grade'].astype(str)
gpkg_path = PROC / 'weather_risk_grid.gpkg'
gpkg_gdf.to_file(gpkg_path, driver='GPKG')

# ── CSV (no geometry column) ──────────────────────────────────────────────────
csv_cols = [c for c in grid.columns if c != 'geometry']
csv_path = PROC / 'weather_risk_grid.csv'
pd.DataFrame(grid[csv_cols]).to_csv(csv_path, index=False, encoding='utf-8-sig')

for p in [gpkg_path, csv_path]:
    print(f'  {p.name}: {p.stat().st_size // 1024} KB  ({len(grid)} rows)')


  weather_risk_grid.gpkg: 252 KB  (283 rows)
  weather_risk_grid.csv: 99 KB  (283 rows)


In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams['axes.unicode_minus'] = False
for fname in ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']:
    try:
        fm.findfont(fname, fallback_to_default=False)
        plt.rcParams['font.family'] = fname
        break
    except Exception:
        pass

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f'KMA Weather Risk — {STATION_NAME}  ({period_start} ~ {period_end}, {n_obs:,} obs)',
    fontsize=12
)

# Panel 1: Wind speed distribution
ws_vals = hist_df['ws'].dropna()
axes[0].hist(ws_vals, bins=50, color='#42A5F5', alpha=0.85, edgecolor='white')
axes[0].axvline(WIND_FAIL_MS, color='red', ls='--', lw=1.5,
                label=f'Fail ≥ {WIND_FAIL_MS} m/s  ({P_wind_fail*100:.2f}%)')
axes[0].set_xlabel('Wind Speed (m/s)')
axes[0].set_ylabel('Hours')
axes[0].set_title('Wind Speed Distribution (2025)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Panel 2: Rainfall distribution (rainy hours only)
rn_vals  = hist_df['rn'].dropna()
rn_rainy = rn_vals[rn_vals > 0]
if len(rn_rainy) > 0:
    axes[1].hist(rn_rainy, bins=40, color='#26A69A', alpha=0.85, edgecolor='white')
    axes[1].axvline(RAIN_FAIL_MMH, color='red', ls='--', lw=1.5,
                    label=f'Fail ≥ {RAIN_FAIL_MMH} mm/h  ({P_rain_fail*100:.2f}%)')
    axes[1].legend(fontsize=9)
    axes[1].set_title(f'Rainfall Distribution (rainy hours: {len(rn_rainy):,})')
else:
    axes[1].text(0.5, 0.5, 'No rainy hours\nin observation period',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=11)
    axes[1].set_title('Rainfall Distribution')
axes[1].set_xlabel('Rainfall (mm/h)')
axes[1].set_ylabel('Hours')
axes[1].grid(alpha=0.3)

# Panel 3: Probability summary bar
labels = ['P_wind\nfail', 'P_rain\nfail', 'P_vis\nfail', 'weather\nrisk', 'score\nweather']
values = [P_wind_fail, P_rain_fail, P_vis_fail, weather_risk, score_weather]
colors = ['#EF5350', '#AB47BC', '#FF7043', '#B0BEC5', '#66BB6A']
bars = axes[2].bar(labels, values, color=colors, alpha=0.88, edgecolor='white')
axes[2].set_ylim(0, 1.12)
axes[2].set_ylabel('Probability (0–1)')
axes[2].set_title('Weather Risk Summary')
axes[2].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.02,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
chart_path = PROC / 'weather_risk_chart.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Chart saved: {chart_path.name}  ({chart_path.stat().st_size // 1024} KB)')


Chart saved: weather_risk_chart.png  (106 KB)


In [14]:
print('[Null check — weather_risk_grid]')
NULL_COLS = ['h3_index', 'score_weather', 'weather_risk',
             'ADM_NM', 'GU_NM', 'CSV_ADMI_CD', 'Ds']
for col in NULL_COLS:
    if col in grid.columns:
        nn = grid[col].isna().sum()
        print(f'  {col}: {nn} nulls{"  <-- PROBLEM" if nn > 0 else ""}')
    else:
        print(f'  {col}: COLUMN MISSING')

print()
print('[Fetch diagnostics]')
print(f'  Weather period    : {START_DATE} ~ {END_DATE}')
print(f'  Chunks total      : {n_chunks}')
print(f'  Chunks succeeded  : {ok_chunks}')
print(f'  Chunks failed     : {fail_chunks}')
print(f'  Raw rows fetched  : {before_dedup:,}')
print(f'  Duplicate rows    : {n_dropped_dup:,}')
print(f'  Clean rows (dedup): {len(hist_df):,}')
print(f'  Missing wind (ws) : {n_missing_ws:,}')
print(f'  Neg. rain → 0     : {n_neg_rain:,}')
print(f'  Wind fail hours   : {n_wind_fail:,}  ({P_wind_fail*100:.3f}%)')
print(f'  Rain fail hours   : {n_rain_fail:,}  ({P_rain_fail*100:.3f}%)')
print(f'  Vis assumption    : {vis_note}')
print(f'  H3 rows in output : {len(grid)}')


[Null check — weather_risk_grid]
  h3_index: 0 nulls
  score_weather: 0 nulls
  weather_risk: 0 nulls
  ADM_NM: 0 nulls
  GU_NM: 0 nulls
  CSV_ADMI_CD: 0 nulls
  Ds: 0 nulls

[Fetch diagnostics]
  Weather period    : 2025-01-01 ~ 2025-12-31
  Chunks total      : 12
  Chunks succeeded  : 12
  Chunks failed     : 0
  Raw rows fetched  : 8,760
  Duplicate rows    : 0
  Clean rows (dedup): 8,760
  Missing wind (ws) : 0
  Neg. rain → 0     : 0
  Wind fail hours   : 0  (0.000%)
  Rain fail hours   : 65  (0.742%)
  Vis assumption    : observed (2129 records, threshold < 1.0 km)
  H3 rows in output : 283


In [15]:
print('=' * 65)
print('NB07c — Weather Risk Index (1-Year) — Final Report')
print('=' * 65)
print(f'Weather source     : kma_hourly_observed_1year')
print(f'Station            : {STATION_NAME}  (STN={STATION_ID})')
print(f'Period             : {period_start} ~ {period_end}')
print(f'Observations used  : {n_obs:,}')
print(f'Chunks used        : {ok_chunks}/{n_chunks}')
print()
print(f'P_wind_fail        : {P_wind_fail:.6f}  ({P_wind_fail*100:.3f}%)')
print(f'P_rain_fail        : {P_rain_fail:.6f}  ({P_rain_fail*100:.3f}%)')
print(f'P_visibility_fail  : {P_vis_fail:.4f}  ({P_vis_fail*100:.2f}%)')
print(f'score_weather      : {score_weather:.6f}  ({score_weather*100:.3f}% drone available)')
print(f'weather_risk       : {weather_risk:.6f}')
print()
print(f'H3 rows written    : {len(grid)}')
print(f'API used           : True')
print(f'Key from env var   : {_KEY_FROM_ENV}')
print(f'Current weather    : {"saved" if CURRENT_OK else "unavailable"} (weather_current.json)')
print()
print('Output files:')
for fname in ['weather_history.csv', 'weather_observed_summary.json',
              'weather_risk_grid.gpkg', 'weather_risk_grid.csv',
              'weather_risk_chart.png', 'weather_current.json']:
    p = PROC / fname
    status = f'{p.stat().st_size // 1024} KB' if p.exists() else 'NOT FOUND'
    print(f'  {fname}: {status}')
print()
print('NB09 handoff:')
print(f'  score_weather = {score_weather:.6f}  (continuous 0-1; same for all H3 cells)')
print('  Ra provides spatial differentiation within Seongnam.')


NB07c — Weather Risk Index (1-Year) — Final Report
Weather source     : kma_hourly_observed_1year
Station            : 수원(119)  (STN=119)
Period             : 2025-01-01 ~ 2025-12-31
Observations used  : 8,760
Chunks used        : 12/12

P_wind_fail        : 0.000000  (0.000%)
P_rain_fail        : 0.007420  (0.742%)
P_visibility_fail  : 0.0488  (4.88%)
score_weather      : 0.944093  (94.409% drone available)
weather_risk       : 0.055907

H3 rows written    : 283
API used           : True
Key from env var   : False
Current weather    : saved (weather_current.json)

Output files:
  weather_history.csv: 816 KB
  weather_observed_summary.json: 0 KB
  weather_risk_grid.gpkg: 252 KB
  weather_risk_grid.csv: 99 KB
  weather_risk_chart.png: 106 KB
  weather_current.json: 0 KB

NB09 handoff:
  score_weather = 0.944093  (continuous 0-1; same for all H3 cells)
  Ra provides spatial differentiation within Seongnam.
